In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import pandas as pd

from src.analytics.ratios import (
    net_profit_margin,
    operating_profit_margin,
    return_on_equity,
    return_on_capital_employed,
    return_on_assets,
)

In [2]:
db_path = PROJECT_ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(db_path)

print("Connected successfully.")

Connected successfully.


In [3]:
sample = pd.read_sql("""
SELECT
    p.company_id,
    p.year,
    p.sales,
    p.operating_profit,
    p.other_income,
    p.net_profit,
    p.interest,

    b.equity_capital,
    b.reserves,
    b.borrowings,
    b.investments,
    b.total_assets

FROM profitandloss p

JOIN balancesheet b
ON p.company_id = b.company_id
AND p.year = b.year

WHERE p.company_id = 'ADANIPORTS'

LIMIT 1;
""", conn)

sample

,company_id,year,sales,operating_profit,other_income,net_profit,interest,equity_capital,reserves,borrowings,investments,total_assets
0,ADANIPORTS,Mar 2013,3577.0,2382.0,344.0,1639.0,542.0,401.0,5993.0,11620.0,222.0,21035.0


In [4]:
row = sample.iloc[0]

print("Company :", row.company_id)
print("Year    :", row.year)

print("\nNet Profit Margin")
print(
    net_profit_margin(
        row.net_profit,
        row.sales
    )
)

print("\nOperating Profit Margin")
print(
    operating_profit_margin(
        row.operating_profit,
        row.sales
    )
)

print("\nROE")
print(
    return_on_equity(
        row.net_profit,
        row.equity_capital,
        row.reserves
    )
)

print("\nROCE")
print(
    return_on_capital_employed(
        row.operating_profit,
        row.interest,
        row.equity_capital,
        row.reserves,
        row.borrowings
    )
)

print("\nROA")
print(
    return_on_assets(
        row.net_profit,
        row.total_assets
    )
)

Company : ADANIPORTS
Year    : Mar 2013

Net Profit Margin
45.820519988817445

Operating Profit Margin
66.59211629857423

ROE
25.633406318423525

ROCE
16.23181969579216

ROA
7.791775612075114


In [5]:
profit = pd.read_sql(
    "SELECT * FROM profitandloss",
    conn
)

balance = pd.read_sql(
    "SELECT * FROM balancesheet",
    conn
)

ratio_df = profit.merge(
    balance,
    on=["company_id", "year"],
    how="inner",
    suffixes=("_pl", "_bs")
)

print(ratio_df.shape)

ratio_df.head()

(1151, 26)


,id_pl,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_asset,total_assets
0,61,ABB,Dec 2012,1653.0,1451.0,202.0,12.0,33.0,0.0,19.0,...,21.0,626.0,0.0,260.0,907.0,109.0,1.0,0.0,798.0,907.0
1,62,ABB,Mar 2014,2276.0,2009.0,267.0,12.0,49.0,0.0,22.0,...,21.0,767.0,0.0,351.0,1139.0,98.0,1.0,0.0,1040.0,1139.0
2,63,ABB,Mar 2015,2289.0,1977.0,312.0,14.0,48.0,0.0,15.0,...,21.0,916.0,0.0,436.0,1374.0,96.0,4.0,0.0,1274.0,1374.0
3,64,ABB,Mar 2016,2614.0,2250.0,365.0,14.0,50.0,3.0,14.0,...,21.0,1174.0,0.0,421.0,1616.0,108.0,3.0,0.0,1505.0,1616.0
4,65,ABB,Mar 2017,2903.0,2505.0,398.0,14.0,57.0,2.0,16.0,...,21.0,1366.0,0.0,679.0,2066.0,110.0,6.0,0.0,1950.0,2066.0


In [6]:
ratio_df["net_profit_margin_pct"] = ratio_df.apply(
    lambda row: net_profit_margin(
        row["net_profit"],
        row["sales"]
    ),
    axis=1
)

ratio_df["operating_profit_margin_pct"] = ratio_df.apply(
    lambda row: operating_profit_margin(
        row["operating_profit"],
        row["sales"]
    ),
    axis=1
)

ratio_df["return_on_equity_pct"] = ratio_df.apply(
    lambda row: return_on_equity(
        row["net_profit"],
        row["equity_capital"],
        row["reserves"]
    ),
    axis=1
)

ratio_df["return_on_capital_employed_pct"] = ratio_df.apply(
    lambda row: return_on_capital_employed(
        row["operating_profit"],
        row["interest"],
        row["equity_capital"],
        row["reserves"],
        row["borrowings"]
    ),
    axis=1
)

ratio_df["return_on_assets_pct"] = ratio_df.apply(
    lambda row: return_on_assets(
        row["net_profit"],
        row["total_assets"]
    ),
    axis=1
)

print("Profitability ratios calculated successfully.")

Profitability ratios calculated successfully.


In [7]:
ratio_df[
    [
        "company_id",
        "year",
        "net_profit_margin_pct",
        "operating_profit_margin_pct",
        "return_on_equity_pct",
        "return_on_capital_employed_pct",
        "return_on_assets_pct"
    ]
].head(10)

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,return_on_capital_employed_pct,return_on_assets_pct
0,ABB,Dec 2012,8.771930,12.220206,22.411128,31.221020,15.986770
1,ABB,Mar 2014,8.699473,11.731107,25.126904,33.883249,17.383670
2,ABB,Mar 2015,10.004369,13.630406,24.439701,33.297759,16.666667
3,ABB,Mar 2016,9.755164,13.963275,21.338912,30.794979,15.779703
4,ABB,Mar 2017,9.541853,13.709955,19.971161,28.839221,13.407551
5,ABB,Mar 2018,12.158884,15.918739,23.685765,31.246308,16.597682
6,ABB,Mar 2019,12.231585,16.444686,22.410359,30.229084,15.300918
7,ABB,Mar 2020,14.488151,18.494991,24.393254,29.393707,16.718354
8,ABB,Mar 2021,16.032483,21.392111,26.556495,34.119782,17.994792
9,ABB,Mar 2022,16.262976,22.023204,28.333333,37.045760,18.915720


In [8]:
ratio_df["opm_difference"] = (
    ratio_df["operating_profit_margin_pct"]
    - ratio_df["opm_percentage"]
).abs()

opm_validation = ratio_df[
    ratio_df["opm_difference"] > 1
].copy()

print(f"Total OPM mismatches (>1%): {len(opm_validation)}")

opm_validation[
    [
        "company_id",
        "year",
        "operating_profit_margin_pct",
        "opm_percentage",
        "opm_difference"
    ]
].head(10)

Total OPM mismatches (>1%): 216


,company_id,year,operating_profit_margin_pct,opm_percentage,opm_difference
22,ADANIENSOL,Mar 2024,34.389113,30.0,4.389113
145,AXISBANK,Mar 2013,30.581614,1353.0,1322.418386
146,AXISBANK,Mar 2014,31.474169,2307.0,2275.525831
147,AXISBANK,Mar 2015,31.362214,3097.0,3065.637786
148,AXISBANK,Mar 2016,32.611984,3466.0,3433.388016
149,AXISBANK,Mar 2017,53.450676,-5715.0,5768.450676
150,AXISBANK,Mar 2018,63.117082,-10277.0,10340.117082
151,AXISBANK,Mar 2019,49.385298,-5447.0,5496.385298
152,AXISBANK,Mar 2020,55.984673,-9859.0,9914.984673
153,AXISBANK,Mar 2021,50.119976,-2510.0,2560.119976


In [9]:
sectors = pd.read_sql(
    """
    SELECT company_id, broad_sector
    FROM sectors
    """,
    conn
)

ratio_df = ratio_df.merge(
    sectors,
    on="company_id",
    how="left"
)

ratio_df[["company_id", "broad_sector"]].head()

,company_id,broad_sector
0,ABB,Industrials
1,ABB,Industrials
2,ABB,Industrials
3,ABB,Industrials
4,ABB,Industrials


In [10]:
opm_validation = ratio_df[
    ratio_df["opm_difference"] > 1
].copy()

In [11]:
opm_validation.groupby("broad_sector").size().sort_values(ascending=False)

broad_sector
Financials                143
Materials                  24
Consumer Discretionary     23
Healthcare                 13
Consumer Staples           12
Energy                      1
dtype: int64

In [12]:
opm_log = opm_validation[
    opm_validation["broad_sector"] != "Financials"
].copy()

print(f"Financial sector mismatches skipped : {len(opm_validation) - len(opm_log)}")
print(f"Mismatches to review                : {len(opm_log)}")

opm_log[
    [
        "company_id",
        "year",
        "broad_sector",
        "operating_profit_margin_pct",
        "opm_percentage",
        "opm_difference"
    ]
].head(10)

Financial sector mismatches skipped : 143
Mismatches to review                : 73


,company_id,year,broad_sector,operating_profit_margin_pct,opm_percentage,opm_difference
22,ADANIENSOL,Mar 2024,Energy,34.389113,30.0,4.389113
299,CIPLA,Mar 2013,Healthcare,73.257640,2214.0,2140.742360
300,CIPLA,Mar 2014,Healthcare,78.895115,2147.0,2068.104885
301,CIPLA,Mar 2015,Healthcare,80.943147,2163.0,2082.056853
302,CIPLA,Mar 2016,Healthcare,82.015954,2480.0,2397.984046
303,CIPLA,Mar 2017,Healthcare,82.659441,2496.0,2413.340559
304,CIPLA,Mar 2018,Healthcare,81.347321,2826.0,2744.652679
305,CIPLA,Mar 2019,Healthcare,81.071996,3097.0,3015.928004
306,CIPLA,Mar 2020,Healthcare,81.286481,3206.0,3124.713519
307,CIPLA,Mar 2021,Healthcare,77.802714,4252.0,4174.197286


In [13]:
opm_log.to_csv(
    PROJECT_ROOT / "output" / "opm_validation_log.csv",
    index=False
)

print("OPM validation log saved.")

OPM validation log saved.


In [14]:
ratio_df["broad_sector"].value_counts()

broad_sector
Financials                294
Energy                    170
Consumer Discretionary    165
Industrials               129
Materials                 118
Consumer Staples           84
Healthcare                 72
Information Technology     71
Communication Services     24
Real Estate                24
Name: count, dtype: int64

In [15]:
ratio_df["roce_benchmark"] = ratio_df["broad_sector"].apply(
    lambda sector: "Sector Relative"
    if sector == "Financials"
    else "Absolute"
)

ratio_df[
    ["company_id",
     "broad_sector",
     "return_on_capital_employed_pct",
     "roce_benchmark"]
].head(15)

,company_id,broad_sector,return_on_capital_employed_pct,roce_benchmark
0,ABB,Industrials,31.221020,Absolute
1,ABB,Industrials,33.883249,Absolute
2,ABB,Industrials,33.297759,Absolute
3,ABB,Industrials,30.794979,Absolute
4,ABB,Industrials,28.839221,Absolute
5,ABB,Industrials,31.246308,Absolute
6,ABB,Industrials,30.229084,Absolute
7,ABB,Industrials,29.393707,Absolute
8,ABB,Industrials,34.119782,Absolute
9,ABB,Industrials,37.045760,Absolute


In [16]:
import importlib
import src.analytics.ratios as ratios

importlib.reload(ratios)

<module 'src.analytics.ratios' from 'c:\\Users\\panka\\OneDrive\\Desktop\\Nifty100_Project\\src\\analytics\\ratios.py'>

In [17]:
row = sample.iloc[0]

In [18]:
print("Company :", row.company_id)
print("Year    :", row.year)

print("\nDebt to Equity")
print(
    ratios.debt_to_equity(
        row.borrowings,
        row.equity_capital,
        row.reserves
    )
)

print("\nInterest Coverage Ratio")
print(
    ratios.interest_coverage_ratio(
        row.operating_profit,
        row.other_income,
        row.interest
    )
)

print("\nNet Debt")
print(
    ratios.net_debt(
        row.borrowings,
        row.investments
    )
)

print("\nAsset Turnover")
print(
    ratios.asset_turnover(
        row.sales,
        row.total_assets
    )
)

Company : ADANIPORTS
Year    : Mar 2013

Debt to Equity
1.817328745699093

Interest Coverage Ratio
5.029520295202952

Net Debt
11398.0

Asset Turnover
0.17004991680532447


In [19]:
ratio_df["debt_to_equity"] = ratio_df.apply(
    lambda row: ratios.debt_to_equity(
        row.borrowings,
        row.equity_capital,
        row.reserves
    ),
    axis=1
)

ratio_df["interest_coverage"] = ratio_df.apply(
    lambda row: ratios.interest_coverage_ratio(
        row.operating_profit,
        row.other_income,
        row.interest
    ),
    axis=1
)

ratio_df["net_debt"] = ratio_df.apply(
    lambda row: ratios.net_debt(
        row.borrowings,
        row.investments
    ),
    axis=1
)

ratio_df["asset_turnover"] = ratio_df.apply(
    lambda row: ratios.asset_turnover(
        row.sales,
        row.total_assets
    ),
    axis=1
)

print("Leverage & Efficiency KPIs calculated successfully.")

Leverage & Efficiency KPIs calculated successfully.


In [20]:
ratio_df[
    [
        "company_id",
        "year",
        "debt_to_equity",
        "interest_coverage",
        "net_debt",
        "asset_turnover"
    ]
].head(10)

,company_id,year,debt_to_equity,interest_coverage,net_debt,asset_turnover
0,ABB,Dec 2012,0.000000,NaN,0.0,1.822492
1,ABB,Mar 2014,0.000000,NaN,0.0,1.998244
2,ABB,Mar 2015,0.000000,NaN,0.0,1.665939
3,ABB,Mar 2016,0.000000,138.333333,0.0,1.617574
4,ABB,Mar 2017,0.000000,227.500000,0.0,1.405131
5,ABB,Mar 2018,0.000000,160.500000,0.0,1.365066
6,ABB,Mar 2019,0.000000,359.000000,0.0,1.250935
7,ABB,Mar 2020,0.071987,96.777778,175.0,1.153933
8,ABB,Mar 2021,0.058801,55.722222,153.0,1.122396
9,ABB,Mar 2022,0.053901,61.315789,152.0,1.163116


In [21]:
ratio_df["high_leverage_flag"] = (
    (ratio_df["debt_to_equity"] > 5) &
    (ratio_df["broad_sector"] != "Financials")
)

ratio_df["icr_label"] = ratio_df["interest_coverage"].apply(
    lambda x: "Debt Free" if pd.isna(x) else None
)

ratio_df["icr_warning"] = ratio_df["interest_coverage"].apply(
    lambda x: False if pd.isna(x) else x < 1.5
)

print("Leverage flags created.")

Leverage flags created.


In [22]:
import importlib
import src.analytics.cagr as cagr

importlib.reload(cagr)

<module 'src.analytics.cagr' from 'c:\\Users\\panka\\OneDrive\\Desktop\\Nifty100_Project\\src\\analytics\\cagr.py'>

In [23]:
print(cagr.calculate_cagr(100, 200, 5))
print(cagr.calculate_cagr(100, -20, 5))
print(cagr.calculate_cagr(-50, 120, 5))
print(cagr.calculate_cagr(0, 120, 5))

(14.869835499703509, 'NORMAL')
(None, 'DECLINE_TO_LOSS')
(None, 'TURNAROUND')
(None, 'ZERO_BASE')


In [24]:
profit_history = pd.read_sql("""
SELECT
    company_id,
    year,
    sales,
    net_profit,
    eps
FROM profitandloss
ORDER BY company_id, year
""", conn)

profit_history.head(20)

,company_id,year,sales,net_profit,eps
0,ABB,Dec 2012,1653.0,145.0,68.0
1,ABB,Mar 2014,2276.0,198.0,93.0
2,ABB,Mar 2015,2289.0,229.0,108.0
3,ABB,Mar 2016,2614.0,255.0,120.0
4,ABB,Mar 2017,2903.0,277.0,130.0
5,ABB,Mar 2018,3298.0,401.0,189.0
6,ABB,Mar 2019,3679.0,450.0,212.0
7,ABB,Mar 2020,4093.0,593.0,279.0
8,ABB,Mar 2021,4310.0,691.0,325.0
9,ABB,Mar 2022,4913.0,799.0,376.0


In [25]:
profit_history.groupby("company_id").size().describe()

count    92.000000
mean     12.793478
std       1.969761
min       3.000000
25%      13.000000
50%      13.000000
75%      13.000000
max      26.000000
dtype: float64

In [26]:
profit_history[
    profit_history["year"].str.extract(r'(\d{4})')[0].isna()
][["company_id", "year"]]


,company_id,year
12,ABB,TTM
24,ADANIENSOL,TTM
37,ADANIENT,TTM
46,ADANIGREEN,TTM
71,ADANIPORTS,TTM
...,...,...
1124,TECHM,TTM
1137,TITAN,TTM
1150,TORNTPHARM,TTM
1163,TRENT,TTM


In [27]:
profit_history = profit_history[
    profit_history["year"] != "TTM"
].copy()

In [28]:
profit_history["year_num"] = (
    profit_history["year"]
    .str.extract(r'(\d{4})')[0]
    .astype(int)
)

profit_history = (
    profit_history
    .sort_values(["company_id", "year_num"])
    .reset_index(drop=True)
)

profit_history[
    ["company_id", "year", "year_num"]
].head(20)

,company_id,year,year_num
0,ABB,Dec 2012,2012
1,ABB,Mar 2014,2014
2,ABB,Mar 2015,2015
3,ABB,Mar 2016,2016
4,ABB,Mar 2017,2017
5,ABB,Mar 2018,2018
6,ABB,Mar 2019,2019
7,ABB,Mar 2020,2020
8,ABB,Mar 2021,2021
9,ABB,Mar 2022,2022


In [29]:
revenue_cagr_results = []

for company, group in profit_history.groupby("company_id"):

    group = group.sort_values("year_num").reset_index(drop=True)

    group["revenue_cagr_5yr"] = None
    group["revenue_cagr_5yr_flag"] = "INSUFFICIENT"

    for i in range(len(group)):

        current_year = group.loc[i, "year_num"]

        previous = group[group["year_num"] == current_year - 5]

        if previous.empty:
            continue

        start_sales = previous.iloc[0]["sales"]
        end_sales = group.loc[i, "sales"]

        cagr_value, flag = cagr.calculate_cagr(
            start_sales,
            end_sales,
            5
        )

        group.loc[i, "revenue_cagr_5yr"] = cagr_value
        group.loc[i, "revenue_cagr_5yr_flag"] = flag

    revenue_cagr_results.append(group)

revenue_cagr_df = pd.concat(
    revenue_cagr_results,
    ignore_index=True
)

print(revenue_cagr_df.shape)

revenue_cagr_df[
    [
        "company_id",
        "year",
        "sales",
        "revenue_cagr_5yr",
        "revenue_cagr_5yr_flag"
    ]
].head(20)

(1085, 8)


,company_id,year,sales,revenue_cagr_5yr,revenue_cagr_5yr_flag
0,ABB,Dec 2012,1653.0,None,INSUFFICIENT
1,ABB,Mar 2014,2276.0,None,INSUFFICIENT
2,ABB,Mar 2015,2289.0,None,INSUFFICIENT
3,ABB,Mar 2016,2614.0,None,INSUFFICIENT
4,ABB,Mar 2017,2903.0,11.921839,NORMAL
5,ABB,Mar 2018,3298.0,None,INSUFFICIENT
6,ABB,Mar 2019,3679.0,10.080782,NORMAL
7,ABB,Mar 2020,4093.0,12.325715,NORMAL
8,ABB,Mar 2021,4310.0,10.518336,NORMAL
9,ABB,Mar 2022,4913.0,11.09639,NORMAL


In [30]:
revenue_cagr_3_results = []

for company, group in profit_history.groupby("company_id"):

    group = group.sort_values("year_num").reset_index(drop=True)

    group["revenue_cagr_3yr"] = None
    group["revenue_cagr_3yr_flag"] = "INSUFFICIENT"

    for i in range(len(group)):

        current_year = group.loc[i, "year_num"]

        previous = group[group["year_num"] == current_year - 3]

        if previous.empty:
            continue

        start_sales = previous.iloc[0]["sales"]
        end_sales = group.loc[i, "sales"]

        cagr_value, flag = cagr.calculate_cagr(
            start_sales,
            end_sales,
            3
        )

        group.loc[i, "revenue_cagr_3yr"] = cagr_value
        group.loc[i, "revenue_cagr_3yr_flag"] = flag

    revenue_cagr_3_results.append(group)

revenue_cagr_3_df = pd.concat(
    revenue_cagr_3_results,
    ignore_index=True
)

print(revenue_cagr_3_df.shape)

revenue_cagr_3_df[
    [
        "company_id",
        "year",
        "sales",
        "revenue_cagr_3yr",
        "revenue_cagr_3yr_flag"
    ]
].head(20)

(1085, 8)


,company_id,year,sales,revenue_cagr_3yr,revenue_cagr_3yr_flag
0,ABB,Dec 2012,1653.0,None,INSUFFICIENT
1,ABB,Mar 2014,2276.0,None,INSUFFICIENT
2,ABB,Mar 2015,2289.0,11.461354,NORMAL
3,ABB,Mar 2016,2614.0,None,INSUFFICIENT
4,ABB,Mar 2017,2903.0,8.448844,NORMAL
5,ABB,Mar 2018,3298.0,12.945332,NORMAL
6,ABB,Mar 2019,3679.0,12.066223,NORMAL
7,ABB,Mar 2020,4093.0,12.132517,NORMAL
8,ABB,Mar 2021,4310.0,9.33072,NORMAL
9,ABB,Mar 2022,4913.0,10.121552,NORMAL


In [31]:
revenue_cagr_10_results = []

for company, group in profit_history.groupby("company_id"):

    group = group.sort_values("year_num").reset_index(drop=True)

    group["revenue_cagr_10yr"] = None
    group["revenue_cagr_10yr_flag"] = "INSUFFICIENT"

    for i in range(len(group)):

        current_year = group.loc[i, "year_num"]

        previous = group[group["year_num"] == current_year - 10]

        if previous.empty:
            continue

        start_sales = previous.iloc[0]["sales"]
        end_sales = group.loc[i, "sales"]

        cagr_value, flag = cagr.calculate_cagr(
            start_sales,
            end_sales,
            10
        )

        group.loc[i, "revenue_cagr_10yr"] = cagr_value
        group.loc[i, "revenue_cagr_10yr_flag"] = flag

    revenue_cagr_10_results.append(group)

revenue_cagr_10_df = pd.concat(
    revenue_cagr_10_results,
    ignore_index=True
)

print(revenue_cagr_10_df.shape)

revenue_cagr_10_df[
    [
        "company_id",
        "year",
        "sales",
        "revenue_cagr_10yr",
        "revenue_cagr_10yr_flag"
    ]
].head(20)

(1085, 8)


,company_id,year,sales,revenue_cagr_10yr,revenue_cagr_10yr_flag
0,ABB,Dec 2012,1653.0,None,INSUFFICIENT
1,ABB,Mar 2014,2276.0,None,INSUFFICIENT
2,ABB,Mar 2015,2289.0,None,INSUFFICIENT
3,ABB,Mar 2016,2614.0,None,INSUFFICIENT
4,ABB,Mar 2017,2903.0,None,INSUFFICIENT
5,ABB,Mar 2018,3298.0,None,INSUFFICIENT
6,ABB,Mar 2019,3679.0,None,INSUFFICIENT
7,ABB,Mar 2020,4093.0,None,INSUFFICIENT
8,ABB,Mar 2021,4310.0,None,INSUFFICIENT
9,ABB,Mar 2022,4913.0,11.50835,NORMAL


In [32]:
revenue_cagr_10_results = []

for company, group in profit_history.groupby("company_id"):

    group = group.sort_values("year_num").reset_index(drop=True)

    group["revenue_cagr_10yr"] = None
    group["revenue_cagr_10yr_flag"] = "INSUFFICIENT"

    for i in range(len(group)):

        current_year = group.loc[i, "year_num"]

        previous = group[group["year_num"] == current_year - 10]

        if previous.empty:
            continue

        start_sales = previous.iloc[0]["sales"]
        end_sales = group.loc[i, "sales"]

        cagr_value, flag = cagr.calculate_cagr(
            start_sales,
            end_sales,
            10
        )

        group.loc[i, "revenue_cagr_10yr"] = cagr_value
        group.loc[i, "revenue_cagr_10yr_flag"] = flag

    revenue_cagr_10_results.append(group)

revenue_cagr_10_df = pd.concat(
    revenue_cagr_10_results,
    ignore_index=True
)

print(revenue_cagr_10_df.shape)

revenue_cagr_10_df[
    [
        "company_id",
        "year",
        "sales",
        "revenue_cagr_10yr",
        "revenue_cagr_10yr_flag"
    ]
].head(20)

(1085, 8)


,company_id,year,sales,revenue_cagr_10yr,revenue_cagr_10yr_flag
0,ABB,Dec 2012,1653.0,None,INSUFFICIENT
1,ABB,Mar 2014,2276.0,None,INSUFFICIENT
2,ABB,Mar 2015,2289.0,None,INSUFFICIENT
3,ABB,Mar 2016,2614.0,None,INSUFFICIENT
4,ABB,Mar 2017,2903.0,None,INSUFFICIENT
5,ABB,Mar 2018,3298.0,None,INSUFFICIENT
6,ABB,Mar 2019,3679.0,None,INSUFFICIENT
7,ABB,Mar 2020,4093.0,None,INSUFFICIENT
8,ABB,Mar 2021,4310.0,None,INSUFFICIENT
9,ABB,Mar 2022,4913.0,11.50835,NORMAL


In [33]:
pat_cagr_5_results = []

for company, group in profit_history.groupby("company_id"):

    group = group.sort_values("year_num").reset_index(drop=True)

    group["pat_cagr_5yr"] = None
    group["pat_cagr_5yr_flag"] = "INSUFFICIENT"

    for i in range(len(group)):

        current_year = group.loc[i, "year_num"]

        previous = group[group["year_num"] == current_year - 5]

        if previous.empty:
            continue

        start_profit = previous.iloc[0]["net_profit"]
        end_profit = group.loc[i, "net_profit"]

        cagr_value, flag = cagr.calculate_cagr(
            start_profit,
            end_profit,
            5
        )

        group.loc[i, "pat_cagr_5yr"] = cagr_value
        group.loc[i, "pat_cagr_5yr_flag"] = flag

    pat_cagr_5_results.append(group)

pat_cagr_5_df = pd.concat(
    pat_cagr_5_results,
    ignore_index=True
)

print(pat_cagr_5_df.shape)

pat_cagr_5_df[
    [
        "company_id",
        "year",
        "net_profit",
        "pat_cagr_5yr",
        "pat_cagr_5yr_flag"
    ]
].head(20)

(1085, 8)


,company_id,year,net_profit,pat_cagr_5yr,pat_cagr_5yr_flag
0,ABB,Dec 2012,145.0,None,INSUFFICIENT
1,ABB,Mar 2014,198.0,None,INSUFFICIENT
2,ABB,Mar 2015,229.0,None,INSUFFICIENT
3,ABB,Mar 2016,255.0,None,INSUFFICIENT
4,ABB,Mar 2017,277.0,13.820989,NORMAL
5,ABB,Mar 2018,401.0,None,INSUFFICIENT
6,ABB,Mar 2019,450.0,17.84454,NORMAL
7,ABB,Mar 2020,593.0,20.960575,NORMAL
8,ABB,Mar 2021,691.0,22.063993,NORMAL
9,ABB,Mar 2022,799.0,23.598557,NORMAL


In [34]:
ratio_df.columns.tolist()

['id_pl',
 'company_id',
 'year',
 'sales',
 'expenses',
 'operating_profit',
 'opm_percentage',
 'other_income',
 'interest',
 'depreciation',
 'profit_before_tax',
 'tax_percentage',
 'net_profit',
 'eps',
 'dividend_payout',
 'id_bs',
 'equity_capital',
 'reserves',
 'borrowings',
 'other_liabilities',
 'total_liabilities',
 'fixed_assets',
 'cwip',
 'investments',
 'other_asset',
 'total_assets',
 'net_profit_margin_pct',
 'operating_profit_margin_pct',
 'return_on_equity_pct',
 'return_on_capital_employed_pct',
 'return_on_assets_pct',
 'opm_difference',
 'broad_sector',
 'roce_benchmark',
 'debt_to_equity',
 'interest_coverage',
 'net_debt',
 'asset_turnover',
 'high_leverage_flag',
 'icr_label',
 'icr_warning']

In [35]:
ratio_df = ratio_df.copy()

ratio_df = ratio_df[
    ratio_df["year"] != "TTM"
].copy()

ratio_df["year_num"] = (
    ratio_df["year"]
    .str.extract(r'(\d{4})')[0]
    .astype(int)
)

ratio_df = ratio_df.sort_values(
    ["company_id", "year_num"]
).reset_index(drop=True)

print(ratio_df.shape)

ratio_df[
    ["company_id", "year", "year_num"]
].head(15)

(1151, 42)


,company_id,year,year_num
0,ABB,Dec 2012,2012
1,ABB,Mar 2014,2014
2,ABB,Mar 2015,2015
3,ABB,Mar 2016,2016
4,ABB,Mar 2017,2017
5,ABB,Mar 2018,2018
6,ABB,Mar 2019,2019
7,ABB,Mar 2020,2020
8,ABB,Mar 2021,2021
9,ABB,Mar 2022,2022


In [36]:
# Revenue, PAT and EPS CAGR (3Y, 5Y, 10Y)

windows = [3, 5, 10]
metrics = {
    "sales": "revenue",
    "net_profit": "pat",
    "eps": "eps"
}

for metric, prefix in metrics.items():

    for years in windows:

        ratio_df[f"{prefix}_cagr_{years}yr"] = None
        ratio_df[f"{prefix}_cagr_{years}yr_flag"] = "INSUFFICIENT"

for company in ratio_df["company_id"].unique():

    company_df = ratio_df[
        ratio_df["company_id"] == company
    ].sort_values("year_num")

    for idx, row in company_df.iterrows():

        current_year = row["year_num"]

        for years in windows:

            previous = company_df[
                company_df["year_num"] == current_year - years
            ]

            if previous.empty:
                continue

            previous = previous.iloc[0]

            for metric, prefix in metrics.items():

                value, flag = cagr.calculate_cagr(
                    previous[metric],
                    row[metric],
                    years
                )

                ratio_df.loc[idx, f"{prefix}_cagr_{years}yr"] = value
                ratio_df.loc[idx, f"{prefix}_cagr_{years}yr_flag"] = flag

In [37]:
ratio_df[
[
    "company_id",
    "year",

    "revenue_cagr_3yr",
    "revenue_cagr_5yr",
    "revenue_cagr_10yr",

    "pat_cagr_3yr",
    "pat_cagr_5yr",
    "pat_cagr_10yr",

    "eps_cagr_3yr",
    "eps_cagr_5yr",
    "eps_cagr_10yr"
]
].head(20)

,company_id,year,revenue_cagr_3yr,revenue_cagr_5yr,revenue_cagr_10yr,pat_cagr_3yr,pat_cagr_5yr,pat_cagr_10yr,eps_cagr_3yr,eps_cagr_5yr,eps_cagr_10yr
0,ABB,Dec 2012,None,None,None,None,None,None,None,None,None
1,ABB,Mar 2014,None,None,None,None,None,None,None,None,None
2,ABB,Mar 2015,11.461354,None,None,16.45438,None,None,16.673336,None,None
3,ABB,Mar 2016,None,None,None,None,None,None,None,None,None
4,ABB,Mar 2017,8.448844,11.921839,None,11.841983,13.820989,None,11.811584,13.837903,None
5,ABB,Mar 2018,12.945332,None,None,20.532167,None,None,20.507113,None,None
6,ABB,Mar 2019,12.066223,10.080782,None,20.843727,17.84454,None,20.888467,17.915415,None
7,ABB,Mar 2020,12.132517,12.325715,None,28.881814,20.960575,None,28.98928,20.902725,None
8,ABB,Mar 2021,9.33072,10.518336,None,19.888601,22.063993,None,19.804699,22.050742,None
9,ABB,Mar 2022,10.121552,11.09639,11.50835,21.090876,23.598557,18.609064,21.046061,23.665597,18.650041


Cash Flow KPIs

In [38]:
cashflow = pd.read_sql("""
SELECT *
FROM cashflow
LIMIT 5
""", conn)

cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0


In [39]:
cashflow.columns.tolist()

['id',
 'company_id',
 'year',
 'operating_activity',
 'investing_activity',
 'financing_activity',
 'net_cash_flow']

In [40]:
cashflow = pd.read_sql("""
SELECT
    company_id,
    year,
    operating_activity,
    investing_activity,
    financing_activity,
    net_cash_flow
FROM cashflow
""", conn)

ratio_df = ratio_df.merge(
    cashflow,
    on=["company_id", "year"],
    how="left"
)

print(ratio_df.shape)

ratio_df[
    [
        "company_id",
        "year",
        "operating_activity",
        "investing_activity",
        "financing_activity"
    ]
].head()

(1174, 64)


,company_id,year,operating_activity,investing_activity,financing_activity
0,ABB,Dec 2012,101.0,-59.0,-42.0
1,ABB,Mar 2014,155.0,-144.0,-42.0
2,ABB,Mar 2014,0.0,0.0,0.0
3,ABB,Mar 2015,215.0,-187.0,-58.0
4,ABB,Mar 2015,-35.0,-1864.0,1902.0


In [41]:
# ---------- Free Cash Flow ----------
ratio_df["free_cash_flow"] = (
    ratio_df["operating_activity"] +
    ratio_df["investing_activity"]
)

# ---------- CFO Quality ----------
ratio_df["cfo_quality_score"] = (
    ratio_df["operating_activity"] /
    ratio_df["net_profit"]
)

ratio_df["cfo_quality_label"] = "Moderate"

ratio_df.loc[
    ratio_df["cfo_quality_score"] > 1,
    "cfo_quality_label"
] = "High Quality"

ratio_df.loc[
    ratio_df["cfo_quality_score"] < 0.5,
    "cfo_quality_label"
] = "Accrual Risk"

# ---------- CapEx Intensity ----------
ratio_df["capex_intensity"] = (
    ratio_df["investing_activity"].abs()
    /
    ratio_df["sales"]
) * 100

ratio_df["capex_label"] = "Moderate"

ratio_df.loc[
    ratio_df["capex_intensity"] < 3,
    "capex_label"
] = "Asset Light"

ratio_df.loc[
    ratio_df["capex_intensity"] > 8,
    "capex_label"
] = "Capital Intensive"

# ---------- FCF Conversion ----------
ratio_df["fcf_conversion"] = (
    ratio_df["free_cash_flow"]
    /
    ratio_df["operating_profit"]
) * 100

ratio_df.loc[
    ratio_df["operating_profit"] == 0,
    "fcf_conversion"
] = None

In [42]:
def capital_pattern(row):

    cfo = "+" if row.operating_activity >= 0 else "-"
    cfi = "+" if row.investing_activity >= 0 else "-"
    cff = "+" if row.financing_activity >= 0 else "-"

    pattern = (cfo, cfi, cff)

    mapping = {
        ("+","-","-"): "Reinvestor",
        ("+","+","-"): "Liquidating Assets",
        ("-","+","+"): "Distress Signal",
        ("-","-","+"): "Growth Funded by Debt",
        ("+","+","+"): "Cash Accumulator",
        ("-","-","-"): "Pre-Revenue",
        ("+","-","+"): "Mixed"
    }

    return mapping.get(pattern, "Other")

ratio_df["capital_allocation_pattern"] = ratio_df.apply(
    capital_pattern,
    axis=1
)

In [43]:
ratio_df[
[
    "company_id",
    "year",
    "free_cash_flow",
    "cfo_quality_score",
    "cfo_quality_label",
    "capex_intensity",
    "capex_label",
    "fcf_conversion",
    "capital_allocation_pattern"
]
].head(20)

,company_id,year,free_cash_flow,cfo_quality_score,cfo_quality_label,capex_intensity,capex_label,fcf_conversion,capital_allocation_pattern
0,ABB,Dec 2012,42.0,0.696552,Moderate,3.569268,Moderate,20.792079,Reinvestor
1,ABB,Mar 2014,11.0,0.782828,Moderate,6.326889,Moderate,4.119850,Reinvestor
2,ABB,Mar 2014,0.0,0.000000,Accrual Risk,0.000000,Asset Light,0.000000,Cash Accumulator
3,ABB,Mar 2015,28.0,0.938865,Moderate,8.169506,Capital Intensive,8.974359,Reinvestor
4,ABB,Mar 2015,-1899.0,-0.152838,Accrual Risk,81.432940,Capital Intensive,-608.653846,Growth Funded by Debt
5,ABB,Mar 2016,172.0,0.976471,Moderate,2.945677,Asset Light,47.123288,Reinvestor
6,ABB,Mar 2016,728.0,6.054902,High Quality,31.216526,Capital Intensive,199.452055,Reinvestor
7,ABB,Mar 2017,152.0,1.108303,High Quality,5.339304,Moderate,38.190955,Reinvestor
8,ABB,Mar 2017,459.0,7.902527,High Quality,59.593524,Capital Intensive,115.326633,Reinvestor
9,ABB,Mar 2018,-62.0,0.381546,Accrual Risk,6.519102,Moderate,-11.809524,Reinvestor


capital_allocation.csv (EXPORTING)

In [44]:
capital_allocation = ratio_df[
    [
        "company_id",
        "year",
        "operating_activity",
        "investing_activity",
        "financing_activity",
        "capital_allocation_pattern"
    ]
].copy()

capital_allocation["cfo_sign"] = capital_allocation["operating_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

capital_allocation["cfi_sign"] = capital_allocation["investing_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

capital_allocation["cff_sign"] = capital_allocation["financing_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

capital_allocation = capital_allocation[
    [
        "company_id",
        "year",
        "cfo_sign",
        "cfi_sign",
        "cff_sign",
        "capital_allocation_pattern"
    ]
]

capital_allocation.to_csv(
    PROJECT_ROOT / "output" / "capital_allocation.csv",
    index=False
)

print("capital_allocation.csv created successfully.")
capital_allocation.head()

capital_allocation.csv created successfully.


,company_id,year,cfo_sign,cfi_sign,cff_sign,capital_allocation_pattern
0,ABB,Dec 2012,+,-,-,Reinvestor
1,ABB,Mar 2014,+,-,-,Reinvestor
2,ABB,Mar 2014,+,+,+,Cash Accumulator
3,ABB,Mar 2015,+,-,-,Reinvestor
4,ABB,Mar 2015,-,-,+,Growth Funded by Debt


financial_ratios populating into sqlite

In [45]:
pd.read_sql("""
PRAGMA table_info(financial_ratios);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,company_id,TEXT,1,None,0
2,2,year,TEXT,1,None,0
3,3,net_profit_margin_pct,REAL,0,None,0
4,4,operating_profit_margin_pct,REAL,0,None,0
5,5,return_on_equity_pct,REAL,0,None,0
6,6,debt_to_equity,REAL,0,None,0
7,7,interest_coverage,REAL,0,None,0
8,8,asset_turnover,REAL,0,None,0
9,9,free_cash_flow_cr,REAL,0,None,0


In [46]:
schema = pd.read_sql("""
PRAGMA table_info(financial_ratios);
""", conn)

schema

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,company_id,TEXT,1,None,0
2,2,year,TEXT,1,None,0
3,3,net_profit_margin_pct,REAL,0,None,0
4,4,operating_profit_margin_pct,REAL,0,None,0
5,5,return_on_equity_pct,REAL,0,None,0
6,6,debt_to_equity,REAL,0,None,0
7,7,interest_coverage,REAL,0,None,0
8,8,asset_turnover,REAL,0,None,0
9,9,free_cash_flow_cr,REAL,0,None,0


In [47]:
duplicates = (
    ratio_df
    .groupby(["company_id", "year"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print("Duplicate company-year pairs:", len(duplicates))
duplicates.head(20)

Duplicate company-year pairs: 83


,company_id,year,count
1,ABB,Mar 2014,2
2,ABB,Mar 2015,2
3,ABB,Mar 2016,2
4,ABB,Mar 2017,2
5,ABB,Mar 2018,2
6,ABB,Mar 2019,2
7,ABB,Mar 2020,2
8,ABB,Mar 2021,2
9,ABB,Mar 2022,2
10,ABB,Mar 2023,2


In [48]:
schema["name"].tolist()

['id',
 'company_id',
 'year',
 'net_profit_margin_pct',
 'operating_profit_margin_pct',
 'return_on_equity_pct',
 'debt_to_equity',
 'interest_coverage',
 'asset_turnover',
 'free_cash_flow_cr',
 'capex_cr',
 'earnings_per_share',
 'book_value_per_share',
 'dividend_payout_ratio_pct',
 'total_debt_cr',
 'cash_from_operations_cr',
 'revenue_cagr_3yr',
 'revenue_cagr_5yr',
 'revenue_cagr_10yr',
 'pat_cagr_3yr',
 'pat_cagr_5yr',
 'pat_cagr_10yr',
 'eps_cagr_3yr',
 'eps_cagr_5yr',
 'eps_cagr_10yr',
 'composite_quality_score']

In [49]:
print("Profit & Loss duplicates")
pd.read_sql("""
SELECT company_id, year, COUNT(*) AS cnt
FROM profitandloss
GROUP BY company_id, year
HAVING COUNT(*) > 1
LIMIT 20;
""", conn)

Profit & Loss duplicates


,company_id,year,cnt
0,ADANIPORTS,Mar 2013,2
1,ADANIPORTS,Mar 2014,2
2,ADANIPORTS,Mar 2015,2
3,ADANIPORTS,Mar 2016,2
4,ADANIPORTS,Mar 2017,2
5,ADANIPORTS,Mar 2018,2
6,ADANIPORTS,Mar 2019,2
7,ADANIPORTS,Mar 2020,2
8,ADANIPORTS,Mar 2021,2
9,ADANIPORTS,Mar 2022,2


In [50]:
print("Balance Sheet duplicates")
pd.read_sql("""
SELECT company_id, year, COUNT(*) AS cnt
FROM balancesheet
GROUP BY company_id, year
HAVING COUNT(*) > 1
LIMIT 20;
""", conn)

Balance Sheet duplicates


,company_id,year,cnt
0,ASIANPAINT,Mar 2013,2
1,ASIANPAINT,Mar 2014,2
2,ASIANPAINT,Mar 2015,2
3,ASIANPAINT,Mar 2016,2
4,ASIANPAINT,Mar 2017,2
5,ASIANPAINT,Mar 2018,2
6,ASIANPAINT,Mar 2019,2
7,ASIANPAINT,Mar 2020,2
8,ASIANPAINT,Mar 2021,2
9,ASIANPAINT,Mar 2022,2


In [51]:
print("Cashflow duplicates")
pd.read_sql("""
SELECT company_id, year, COUNT(*) AS cnt
FROM cashflow
GROUP BY company_id, year
HAVING COUNT(*) > 1
LIMIT 20;
""", conn)

Cashflow duplicates


,company_id,year,cnt
0,ABB,Mar 2014,2
1,ABB,Mar 2015,2
2,ABB,Mar 2016,2
3,ABB,Mar 2017,2
4,ABB,Mar 2018,2
5,ABB,Mar 2019,2
6,ABB,Mar 2020,2
7,ABB,Mar 2021,2
8,ABB,Mar 2022,2
9,ABB,Mar 2023,2


In [52]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,companies
1,profitandloss
2,balancesheet
3,cashflow
4,analysis
5,documents
6,prosandcons
7,financial_ratios
8,market_cap
9,peer_groups


In [53]:
pd.read_sql("""
SELECT company_id, year, COUNT(*) AS cnt
FROM profitandloss
GROUP BY company_id, year
HAVING COUNT(*) > 1;
""", conn)

,company_id,year,cnt
0,ADANIPORTS,Mar 2013,2
1,ADANIPORTS,Mar 2014,2
2,ADANIPORTS,Mar 2015,2
3,ADANIPORTS,Mar 2016,2
4,ADANIPORTS,Mar 2017,2
5,ADANIPORTS,Mar 2018,2
6,ADANIPORTS,Mar 2019,2
7,ADANIPORTS,Mar 2020,2
8,ADANIPORTS,Mar 2021,2
9,ADANIPORTS,Mar 2022,2


In [54]:
pd.read_sql("""
SELECT company_id, year, COUNT(*) AS cnt
FROM balancesheet
GROUP BY company_id, year
HAVING COUNT(*) > 1;
""", conn)

,company_id,year,cnt
0,ASIANPAINT,Mar 2013,2
1,ASIANPAINT,Mar 2014,2
2,ASIANPAINT,Mar 2015,2
3,ASIANPAINT,Mar 2016,2
4,ASIANPAINT,Mar 2017,2
5,ASIANPAINT,Mar 2018,2
6,ASIANPAINT,Mar 2019,2
7,ASIANPAINT,Mar 2020,2
8,ASIANPAINT,Mar 2021,2
9,ASIANPAINT,Mar 2022,2


In [55]:
pd.read_sql("""
SELECT company_id, year, COUNT(*) AS cnt
FROM cashflow
GROUP BY company_id, year
HAVING COUNT(*) > 1;
""", conn)

,company_id,year,cnt
0,ABB,Mar 2014,2
1,ABB,Mar 2015,2
2,ABB,Mar 2016,2
3,ABB,Mar 2017,2
4,ABB,Mar 2018,2
5,ABB,Mar 2019,2
6,ABB,Mar 2020,2
7,ABB,Mar 2021,2
8,ABB,Mar 2022,2
9,ABB,Mar 2023,2


In [56]:
print("Before:", ratio_df.shape)

ratio_df = (
    ratio_df
    .sort_values(["company_id", "year"])
    .drop_duplicates(
        subset=["company_id", "year"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("After:", ratio_df.shape)

ratio_df[
    ["company_id", "year"]
].head()

Before: (1174, 71)
After: (1055, 71)


,company_id,year
0,ABB,Dec 2012
1,ABB,Mar 2014
2,ABB,Mar 2015
3,ABB,Mar 2016
4,ABB,Mar 2017


In [57]:
duplicates = (
    ratio_df
    .groupby(["company_id", "year"])
    .size()
)

print("Duplicate keys:", (duplicates > 1).sum())

Duplicate keys: 0


In [58]:
pd.read_sql("""
SELECT COUNT(*)
FROM financial_ratios;
""", conn)

,COUNT(*)
0,1160


In [59]:
pd.read_sql("""
SELECT
    company_id,
    year,
    net_profit_margin_pct,
    operating_profit_margin_pct,
    revenue_cagr_5yr
FROM financial_ratios
LIMIT 10;
""", conn)

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,revenue_cagr_5yr
0,ABB,Dec 2012,8.77,12.0,None
1,ABB,Mar 2014,8.70,12.0,None
2,ABB,Mar 2014,8.70,12.0,None
3,ABB,Mar 2015,10.00,14.0,None
4,ABB,Mar 2015,10.00,14.0,None
5,ABB,Mar 2016,9.76,14.0,None
6,ABB,Mar 2016,9.76,14.0,None
7,ABB,Mar 2017,9.54,14.0,None
8,ABB,Mar 2017,9.54,14.0,None
9,ABB,Mar 2018,12.16,16.0,None


In [60]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DB_PATH = PROJECT_ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(DB_PATH)

columns = pd.read_sql(
    "PRAGMA table_info(financial_ratios);",
    conn
)

conn.close()

print(columns["name"].tolist())

['id', 'company_id', 'year', 'net_profit_margin_pct', 'operating_profit_margin_pct', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'free_cash_flow_cr', 'capex_cr', 'earnings_per_share', 'book_value_per_share', 'dividend_payout_ratio_pct', 'total_debt_cr', 'cash_from_operations_cr', 'revenue_cagr_3yr', 'revenue_cagr_5yr', 'revenue_cagr_10yr', 'pat_cagr_3yr', 'pat_cagr_5yr', 'pat_cagr_10yr', 'eps_cagr_3yr', 'eps_cagr_5yr', 'eps_cagr_10yr', 'composite_quality_score']


In [61]:
for col in columns["name"].tolist():
    print(col)

id
company_id
year
net_profit_margin_pct
operating_profit_margin_pct
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
free_cash_flow_cr
capex_cr
earnings_per_share
book_value_per_share
dividend_payout_ratio_pct
total_debt_cr
cash_from_operations_cr
revenue_cagr_3yr
revenue_cagr_5yr
revenue_cagr_10yr
pat_cagr_3yr
pat_cagr_5yr
pat_cagr_10yr
eps_cagr_3yr
eps_cagr_5yr
eps_cagr_10yr
composite_quality_score


In [62]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/db/nifty100.db")

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

conn.close()

print(tables)

                name
0          companies
1      profitandloss
2       balancesheet
3           cashflow
4           analysis
5          documents
6        prosandcons
7   financial_ratios
8         market_cap
9        peer_groups
10           sectors
11      stock_prices


In [63]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/db/nifty100.db")

market = pd.read_sql(
    "SELECT * FROM market_cap LIMIT 5;",
    conn
)

print(market.head())
print(market.columns.tolist())

conn.close()

   id company_id  year  market_cap_crore  enterprise_value_crore  pe_ratio  \
0   1        ABB  2019         844312.98               814410.48     19.29   
1   2        ABB  2020         923674.33              1120679.72     67.54   
2   3        ABB  2021        1020674.09              1166450.81     62.63   
3   4        ABB  2022        1185994.82              1158637.42     18.68   
4   5        ABB  2023        1161513.28              1103083.77     58.34   

   pb_ratio  ev_ebitda  dividend_yield_pct  
0     14.26      22.84                0.84  
1     13.03      24.93                3.90  
2      6.67       7.58                1.58  
3      4.75       6.12                3.35  
4     12.72       5.70                1.05  
['id', 'company_id', 'year', 'market_cap_crore', 'enterprise_value_crore', 'pe_ratio', 'pb_ratio', 'ev_ebitda', 'dividend_yield_pct']


In [64]:
conn = sqlite3.connect("../data/db/nifty100.db")

companies = pd.read_sql(
    "SELECT * FROM companies LIMIT 5;",
    conn
)

print(companies.head())
print(companies.columns.tolist())

conn.close()

           id                                       company_logo  \
0         ABB   https://mkt.in/static/mkt-icons/nifty100/ABB.png   
1  ADANIENSOL  https://m.economictimes.com/thumb/msid-1173715...   
2    ADANIENT  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
3  ADANIGREEN  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
4  ADANIPORTS  https://mkt.in/static/mkt-icons/nifty100/ADANI...   

                                company_name  \
0                           Abbott India Ltd   
1                 Adani Energy Solutions Ltd   
2                      Adani Enterprises Ltd   
3                     Adani Green Energy Ltd   
4  Adani Ports & Special Economic Zone Ltd\n   

                                          chart_link  \
0  https://in.tradingview.com/chart/?symbol=NSE%3...   
1  https://in.tradingview.com/chart/?symbol=NSE%3...   
2  https://in.tradingview.com/chart/?symbol=ADANIENT   
3  https://in.tradingview.com/chart/?symbol=NSE%3...   
4  https://in.tradingv

In [65]:
conn = sqlite3.connect("../data/db/nifty100.db")

sectors = pd.read_sql(
    "SELECT * FROM sectors LIMIT 5;",
    conn
)

print(sectors.head())
print(sectors.columns.tolist())

conn.close()

   id  company_id broad_sector         sub_sector  index_weight_pct  \
0   1         ABB  Industrials      Capital Goods              0.81   
1   2  ADANIENSOL       Energy  Power & Utilities              0.65   
2   3    ADANIENT  Industrials      Conglomerates              1.50   
3   4  ADANIGREEN       Energy   Renewable Energy              1.23   
4   5  ADANIPORTS  Industrials     Infrastructure              2.12   

  market_cap_category  
0           Large Cap  
1           Large Cap  
2           Large Cap  
3           Large Cap  
4           Large Cap  
['id', 'company_id', 'broad_sector', 'sub_sector', 'index_weight_pct', 'market_cap_category']
